In [2]:
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.7 MB/s eta 0:00:00


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


/tmp/ipykernel_2818/4049625560.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
loader = PyPDFLoader("JAY KUMAR JAIN- PCE23CS072.pdf")
docu = loader.load()
# print(len(docu))
print(docu[0].metadata)



{'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-02T18:24:13+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-02T18:24:13+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'JAY KUMAR JAIN- PCE23CS072.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(docu)
print(len(chunks))
print(chunks[0].metadata)



4
{'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-02T18:24:13+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-02T18:24:13+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'JAY KUMAR JAIN- PCE23CS072.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embedding = model.encode("Hello World")

print(type(embedding))
print(len(embedding))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

<class 'numpy.ndarray'>
384


In [ ]:
# generate embedding for every chunks

In [7]:
texts = [doc.page_content for doc in chunks]
embeddings = model.encode(texts)
print(len(embeddings))

4


In [8]:
# sotres in FAISS

In [9]:
import faiss
import numpy as np

In [10]:
vectors = np.array(embeddings).astype("float32")

In [11]:
dimension = vectors.shape[1]
index = faiss.IndexFlat(dimension)

In [12]:
index.add(vectors)

In [13]:
print(index.ntotal)

4


In [15]:
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q sentence-transformers faiss-cpu pypdf

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [16]:
# retriever pipeline

In [17]:
retriever = vector_store.as_retriever(search_type = "similarity",search_kwargs = {"k":4})

In [18]:
retriever.invoke("What is the professional summary of Jay Kumar Jain?")

[Document(id='b3107afb-93ab-434d-9ece-19dabbd51673', metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-02T18:24:13+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-02T18:24:13+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'JAY KUMAR JAIN- PCE23CS072.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Jay Kumar Jain\n+91-7877984434|jayjainbny123@gmail.com|Jaipur, Rajasthan\nLinkedIn|GitHub\nPROFESSIONAL SUMMAR Y\nSoftware Developer with hands-on experience in full-stack web development, machine learning, and software engi-\nneering projects. Strong foundation in Java, Python, Data Structures, Algorithms, Object-Oriented Programming,\nDBMS, and Software Development Life Cycle. Experienced in designing RESTful APIs, solving analytical problems,\nbuilding scalable applications, and collab

In [19]:
# augmentation part which merge query and prompt

In [20]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are a helpful assistant.

Answer only from the provided context.

If the content is insufficient, just say:
"I don't know."

Context:
{context}

Question:
{question}
""",
    input_variables=["context", "question"]
)


In [21]:
question = "What is the professional summary of Jay Kumar Jain?"
retrieved_docs = retriever.invoke(question)

In [22]:
retrieved_docs

[Document(id='b3107afb-93ab-434d-9ece-19dabbd51673', metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-02T18:24:13+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-02T18:24:13+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'JAY KUMAR JAIN- PCE23CS072.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Jay Kumar Jain\n+91-7877984434|jayjainbny123@gmail.com|Jaipur, Rajasthan\nLinkedIn|GitHub\nPROFESSIONAL SUMMAR Y\nSoftware Developer with hands-on experience in full-stack web development, machine learning, and software engi-\nneering projects. Strong foundation in Java, Python, Data Structures, Algorithms, Object-Oriented Programming,\nDBMS, and Software Development Life Cycle. Experienced in designing RESTful APIs, solving analytical problems,\nbuilding scalable applications, and collab

In [23]:
context_text = "\n\n".join(docu.page_content for docu in retrieved_docs)
context_text

'Jay Kumar Jain\n+91-7877984434|jayjainbny123@gmail.com|Jaipur, Rajasthan\nLinkedIn|GitHub\nPROFESSIONAL SUMMAR Y\nSoftware Developer with hands-on experience in full-stack web development, machine learning, and software engi-\nneering projects. Strong foundation in Java, Python, Data Structures, Algorithms, Object-Oriented Programming,\nDBMS, and Software Development Life Cycle. Experienced in designing RESTful APIs, solving analytical problems,\nbuilding scalable applications, and collaborating in Agile development environments.\nEDUCA TION\nPoornima College of Engineering, Jaipur2023–2027\nB.Tech in Computer Science & Engineering CGPA: 9.23\nRelevant Coursework:Data Structures & Algorithms, Object-Oriented Programming, DBMS, Operating Sys-\ntems, Software Engineering, Computer Networks\nClass XII (CBSE) 2022\nPercentage: 94.6%\nEXPERIENCE\nMERN Stack Intern|IIT RoparOngoing\n•Engineered full-stack web applications using MongoDB, Express.js, React.js, and Node.js\n\n•Analyzed and pro

In [24]:
final_prompt = prompt.invoke({"context":context_text,'question':question})
final_prompt

StringPromptValue(text='\nYou are a helpful assistant.\n\nAnswer only from the provided context.\n\nIf the content is insufficient, just say:\n"I don\'t know."\n\nContext:\nJay Kumar Jain\n+91-7877984434|jayjainbny123@gmail.com|Jaipur, Rajasthan\nLinkedIn|GitHub\nPROFESSIONAL SUMMAR Y\nSoftware Developer with hands-on experience in full-stack web development, machine learning, and software engi-\nneering projects. Strong foundation in Java, Python, Data Structures, Algorithms, Object-Oriented Programming,\nDBMS, and Software Development Life Cycle. Experienced in designing RESTful APIs, solving analytical problems,\nbuilding scalable applications, and collaborating in Agile development environments.\nEDUCA TION\nPoornima College of Engineering, Jaipur2023–2027\nB.Tech in Computer Science & Engineering CGPA: 9.23\nRelevant Coursework:Data Structures & Algorithms, Object-Oriented Programming, DBMS, Operating Sys-\ntems, Software Engineering, Computer Networks\nClass XII (CBSE) 2022\nPerc

In [25]:
# generation part from llm

In [28]:
import os

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6Lh6WQP4o0x28O0z1VyQpqAAfj_C5lvOrLtNLApU4CP4g"

In [29]:
!pip install -q langchain-google-genai google-generativeai
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
ans = llm.invoke(final_prompt)

In [30]:
print(ans.content)

Software Developer with hands-on experience in full-stack web development, machine learning, and software engi- neering projects. Strong foundation in Java, Python, Data Structures, Algorithms, Object-Oriented Programming, DBMS, and Software Development Life Cycle. Experienced in designing RESTful APIs, solving analytical problems, building scalable applications, and collaborating in Agile development environments.


In [31]:
# build a parallel chain

In [32]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser


In [33]:
def format_docs(retrieved_docs):
    context_text='\n\n'.join(doc.page_content for doc in retrieved_docs)
    return context_text

In [34]:
parallel_chain = RunnableParallel({
    "context": retriever| RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [38]:
parallel_chain.invoke("What language did Jay knows")

{'context': 'Jay Kumar Jain\n+91-7877984434|jayjainbny123@gmail.com|Jaipur, Rajasthan\nLinkedIn|GitHub\nPROFESSIONAL SUMMAR Y\nSoftware Developer with hands-on experience in full-stack web development, machine learning, and software engi-\nneering projects. Strong foundation in Java, Python, Data Structures, Algorithms, Object-Oriented Programming,\nDBMS, and Software Development Life Cycle. Experienced in designing RESTful APIs, solving analytical problems,\nbuilding scalable applications, and collaborating in Agile development environments.\nEDUCA TION\nPoornima College of Engineering, Jaipur2023–2027\nB.Tech in Computer Science & Engineering CGPA: 9.23\nRelevant Coursework:Data Structures & Algorithms, Object-Oriented Programming, DBMS, Operating Sys-\ntems, Software Engineering, Computer Networks\nClass XII (CBSE) 2022\nPercentage: 94.6%\nEXPERIENCE\nMERN Stack Intern|IIT RoparOngoing\n•Engineered full-stack web applications using MongoDB, Express.js, React.js, and Node.js\n\n•Anal

In [36]:
# create second chain

In [42]:
parser = StrOutputParser()
main_chain= parallel_chain | prompt | llm | parser
main_chain.invoke("Who languages did jay know in this pdf")

'Programming Languages: Java, Python, JavaScript, C++'

In [41]:
parser = StrOutputParser()
main_chain= parallel_chain | prompt | llm | parser
main_chain.invoke("summarize the pdf")

'Jay Kumar Jain is a Software Developer with experience in full-stack web development, machine learning, and software engineering. He is pursuing a B.Tech in Computer Science & Engineering at Poornima College of Engineering (2023-2027) with a CGPA of 9.23, and achieved 94.6% in Class XII (CBSE).\n\nHis experience includes working as a MERN Stack Intern at IIT Ropar, where he engineered full-stack web applications, architected RESTful APIs, and enhanced application responsiveness. He also completed AI/ML Training at R-CAT Jaipur, constructing machine learning pipelines and performing data preprocessing and model evaluation.\n\nKey projects include a Loan Approval Prediction system (analyzing 10,000+ records, achieving 85%+ accuracy), a full-stack VI Notes Web Application using React.js, and an ongoing AI Online Exam Proctoring System utilizing OpenCV and machine learning for real-time monitoring.\n\nHis technical skills encompass programming languages like Java, Python, JavaScript, and 

In [43]:
parser = StrOutputParser()
main_chain= parallel_chain | prompt | llm | parser
main_chain.invoke("What certifications has Jay Kumar Jain earned?")

'Jay Kumar Jain has earned the following certifications:\n*   NPTEL Programming in Java – Top 5% (Gold)\n*   NPTEL Joy of Computing using Python – Top 2% (Silver)\n*   Google Cloud Career Launchpad – Data Analytics Track'